# Augmentation Ablation: ViT-B/16 at 100% Data
Best-performing combo overall (82.1% test acc with augmentation on). Reruns the same config with `RandomHorizontalFlip` removed to isolate augmentation's effect.

Requires this session to already have `PROJECT_DIR`, `SEED`, `device`, `NUM_CLASSES`, `BATCH_SIZE`, `test_loader`, `make_train_val_loaders`, `train_one_epoch`, `evaluate`, and `run_vit_experiment` defined (i.e. sections 1-8 and the Part A ViT cells from `food101_setup.ipynb` already run in this session).

## 1. Build the no-augmentation dataset
Same as the original `train_transform` but without `RandomHorizontalFlip`, isolating augmentation as the only variable.

In [ ]:
from torchvision import transforms
from torchvision.datasets import Food101

train_transform_no_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_set_no_aug = Food101(root=DATA_ROOT, split='train', transform=train_transform_no_aug, download=False)
train_100_no_aug = train_set_no_aug  # 100% data, no augmentation

print('No-aug train set size:', len(train_100_no_aug))

## 2. Run the ablation
Same config as your best combo: `lr=5e-5, wd=0.0001`. Epoch count is a time/accuracy tradeoff, see note above, adjust `ABLATION_EPOCHS` before running.

In [ ]:
ABLATION_EPOCHS = 5  # matches the original best-combo run exactly (5 epochs, val peaked at epoch 3)
BEST_LR_VIT = 5e-5
BEST_WD_VIT = 1e-4

model_no_aug, ablation_metrics = run_vit_experiment(
    train_100_no_aug,
    fraction_label='100',
    epochs=ABLATION_EPOCHS,
    aug_label='aug_off',
    lr=BEST_LR_VIT,
    wd=BEST_WD_VIT,
)

print('Ablation (augmentation off) results:', ablation_metrics)

## 3. Compare against the augmentation-on baseline

In [ ]:
import json

AUG_ON_TEST_ACC = 0.821  # confirmed baseline, ViT-B/16, 100% data, augmentation on

aug_off_test_acc = ablation_metrics['test_acc']
delta = AUG_ON_TEST_ACC - aug_off_test_acc

print(f'Augmentation ON  test acc: {AUG_ON_TEST_ACC*100:.1f}%')
print(f'Augmentation OFF test acc: {aug_off_test_acc*100:.1f}%')
print(f'Difference (on - off): {delta*100:+.1f} pts')

ablation_results = {
    'model': 'vit_b_16',
    'data_fraction': '100',
    'lr': BEST_LR_VIT,
    'weight_decay': BEST_WD_VIT,
    'epochs': ABLATION_EPOCHS,
    'aug_on_test_acc': AUG_ON_TEST_ACC,
    'aug_off_test_acc': aug_off_test_acc,
    'difference_pts': delta,
}

with open(f'{PROJECT_DIR}/results/augmentation_ablation.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)
print('Saved to', f'{PROJECT_DIR}/results/augmentation_ablation.json')

## For the report
One sentence covers this: "Removing horizontal-flip augmentation on the best-performing configuration (ViT-B/16, 100% data) changed test accuracy from X% to Y%, a difference of Z points", plus a short interpretation, e.g. whether the flip helped generalization or made little difference given the model was already pretrained on a large, diverse dataset.